# Serve MedGemma for the licenta pipeline (Colab Pro)

This notebook serves **MedGemma** behind a vLLM OpenAI-compatible API and exposes it
with a free Cloudflare quick-tunnel, so your **local** CrewAI pipeline can call it via
`LLM_BACKEND=medgemma`.

**Before you start:**
1. Accept the model license on Hugging Face (one-time, free): open
   https://huggingface.co/google/medgemma-4b-it and click *Agree and access*.
2. Create a HF access token (read): https://huggingface.co/settings/tokens
3. In Colab: **Runtime -> Change runtime type -> GPU** (L4 recommended for 4B;
   A100 if you later try 27B). Then **Runtime -> Run all**, or run the cells in order.

Keep this tab open while you run the benchmark locally — when the Colab session ends,
the tunnel URL dies and you start a fresh one (the URL changes each session).

In [ ]:
# 1) Check the GPU you got (L4/A100 ideal; T4 works for 4B but is slower).
!nvidia-smi

In [ ]:
# 2) Install vLLM + a MATCHING torch, via uv (vLLM's recommended installer).
#    Colab's GPU driver is CUDA 13.0, and the latest vLLM wheel is CUDA-13-only.
#    So we go ALL cu13: --torch-backend=auto detects the 13.0 driver and installs the
#    cu13 torch that ships libcudart.so.13 (what vllm._C needs). This avoids both:
#      * libcudart.so.13 missing  (happens if torch is left at cu12.8), and
#      * old-vLLM 'rope_scaling' config errors (we keep the LATEST vLLM, which knows MedGemma).
#    If 'auto' ever lands on cu12, replace it with: --torch-backend=cu130
!pip -q install -U uv
!uv pip install --system -q -U vllm --torch-backend=auto
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print('done -> now do Runtime > Restart session, then run cells 3 and 4 (skip cell 2).')

In [ ]:
# 3) Log in to Hugging Face (MedGemma is a gated model).
import getpass
from huggingface_hub import login
login(token=getpass.getpass('Paste your HF token (read): '))
print('logged in')

In [ ]:
# 4) Start the vLLM OpenAI-compatible server and STREAM its log live.
#    Exits immediately if the server crashes (you'll see the traceback right away),
#    announces when it's ready, and caps the wait at 20 min (first run downloads ~8 GB).
#    4B fits L4/T4. For 27B see the optional cell at the bottom.
import subprocess, time, requests

MODEL    = 'google/medgemma-4b-it'   # served model id (must match local MEDGEMMA_MODEL)
MAX_LEN  = 8192                       # context length; lower it (e.g. 4096) if you hit OOM

logf = open('vllm.log', 'w')
server = subprocess.Popen(
    ['vllm', 'serve', MODEL,
     '--host', '0.0.0.0', '--port', '8000',
     '--max-model-len', str(MAX_LEN),
     '--dtype', 'bfloat16',
     '--gpu-memory-utilization', '0.92'],
    stdout=logf, stderr=subprocess.STDOUT)

print(f'Starting vLLM for {MODEL} — streaming log below...\n')
reader = open('vllm.log', 'r')
ready = False
deadline = time.time() + 1200   # 20 min cap
while time.time() < deadline:
    chunk = reader.read()       # echo any new log lines as they appear
    if chunk:
        print(chunk, end='')
    if server.poll() is not None:               # server died -> show the rest + stop
        print(reader.read(), end='')
        print(f'\n\n[!] vLLM exited early (code {server.returncode}) — see the traceback above.')
        break
    try:                                        # ready?
        if requests.get('http://localhost:8000/v1/models', timeout=2).ok:
            print(reader.read(), end='')
            print('\n\n[OK] vLLM ready — run cell 5 to open the tunnel.')
            ready = True
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print('\n\n[!] Timed out after 20 min — check the streamed log above.')

In [ ]:
# 5) Open a public Cloudflare quick-tunnel to the server and print the URL.
import subprocess, re, time

tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=open('cf.log', 'w'), stderr=subprocess.STDOUT)

public_url = None
for _ in range(40):
    time.sleep(2)
    try:
        m = re.search(r'https://[-\w.]+trycloudflare\.com', open('cf.log').read())
        if m:
            public_url = m.group(0); break
    except FileNotFoundError:
        pass

print('=' * 70)
if public_url:
    print('PUBLIC URL :', public_url)
    print()
    print('Put these in your LOCAL .env (note the /v1 suffix):')
    print('  LLM_BACKEND=medgemma')
    print('  MEDGEMMA_BASE_URL=' + public_url + '/v1')
    print('  MEDGEMMA_MODEL=' + MODEL)
    print('  MEDGEMMA_API_KEY=EMPTY')
else:
    print('No URL yet - re-run this cell, or check cf.log')
print('=' * 70)

In [ ]:
# 6) Sanity check: a JSON-extraction prompt like the parser uses.
from openai import OpenAI
client = OpenAI(base_url='http://localhost:8000/v1', api_key='EMPTY')
resp = client.chat.completions.create(
    model=MODEL, temperature=0,
    messages=[{'role': 'user', 'content':
        'Return JSON with keys age, chief_complaints, pain_score for: '
        "I'm 64, crushing chest pain for an hour, pain is 9 out of 10."}])
print(resp.choices[0].message.content)

## What to do after the URL prints

On your **local machine** (this repo), edit `.env`:
```
LLM_BACKEND=medgemma
MEDGEMMA_BASE_URL=https://<your>.trycloudflare.com/v1
MEDGEMMA_MODEL=google/medgemma-4b-it
MEDGEMMA_API_KEY=EMPTY
```
Then run (keep this Colab tab alive the whole time):
```bash
# Smoke test (1 case, proves the plumbing + parser bypass):
uv run python benchmarks/benchmark_pipeline_e2e.py --limit 1 --llm-backend medgemma --parser-llm --skip-feature-vector

# Experiment A - full 20-case parser/pipeline comparison:
uv run python benchmarks/benchmark_pipeline_e2e.py --llm-backend medgemma --parser-llm --dump-json artifacts/benchmarks/e2e_medgemma.json
#   (and the Flash baseline, no server needed:)
uv run python benchmarks/benchmark_pipeline_e2e.py --dump-json artifacts/benchmarks/e2e_flash.json

# Experiment B - case-generation narrative quality:
uv run generate_cases --backend medgemma --limit 5
uv run python benchmarks/compare_case_generation.py
```
When done, set `LLM_BACKEND=flash` again locally and stop this runtime.

In [ ]:
# (Optional) Keep the cell busy so Colab is less likely to idle-disconnect while you
# run the benchmark locally. Interrupt this cell when you're finished.
import time
while True:
    time.sleep(60)

---
## Optional: MedGemma 27B (needs an A100, e.g. Colab Pro+)

27B in bf16 is ~54 GB and will NOT fit one GPU. Use a 4-bit checkpoint and adjust cell 4:
```python
MODEL = 'google/medgemma-27b-text-it'   # or a community AWQ/GPTQ 4-bit repo
# add to the vllm serve args:  '--quantization', 'awq',  '--max-model-len', '4096'
```
If no official 4-bit repo is available, start with 4B (above) — it's the safe, runnable
comparison point. Note in your thesis whether the 27B run was quantized.

## Optional: agentic tool-calling

The `--parser-llm` benchmark mode does **not** need tool-calling. If you also want to try
MedGemma driving the *full agentic crew* (drop `--parser-llm`), add to cell 4's serve args:
```python
'--enable-auto-tool-choice', '--tool-call-parser', 'pythonic'
```
and re-run. If the crew still fails to call tools, that's a legitimate finding — fall back
to `--parser-llm` (MedGemma's evaluation path).